# LeetCode 2280 — Minimum Lines to Represent a Line Chart bằng Python

Link đề: https://leetcode.com/problems/minimum-lines-to-represent-a-line-chart/ · Độ khó: **Medium** · Chủ đề: Array, Math, Geometry, Sorting, Number Theory

Skeleton LeetCode:

```python
class Solution:
    def minimumLines(self, stockPrices: List[List[int]]) -> int:
        ...
```

## 1. Mục tiêu học

1. Nhận ra đồ thị đường gấp khúc chỉ cần **đếm số lần đổi hướng (đổi hệ số góc)** giữa các đoạn liền kề sau khi **sắp xếp theo ngày**.
2. So sánh hệ số góc **chính xác** bằng **nhân chéo** (cross multiplication) — và thấy tận mắt vì sao **so sánh `float` sai** với tọa độ tới 10⁹ (đây là bẫy nổi tiếng của bài).
3. Biểu diễn hệ số góc dạng **phân số tối giản** `(dy/g, dx/g)` hoặc `fractions.Fraction` như các cách chính xác khác.
4. Xử lý biên: `n = 1` ⇒ 0 đường; `n = 2` ⇒ 1 đường.
5. Viết test có dữ liệu "gần thẳng hàng" để bắt lỗi làm tròn.

## 2. Đề bài & ví dụ

`stockPrices[i] = [day_i, price_i]`: giá cổ phiếu ngày `day_i`. Vẽ biểu đồ đường: đặt các điểm lên mặt phẳng (trục X là ngày, Y là giá) và **nối các điểm liền kề** (theo ngày). Trả về **số đoạn thẳng ít nhất** để biểu diễn biểu đồ (các đoạn liền kề cùng hệ số góc gộp thành một đường).

**Ví dụ 1:** `[[1,7],[2,6],[3,5],[4,4],[5,4],[6,3],[7,2],[8,1]]` → `3`: đường `(1,7)–(4,4)`, đường `(4,4)–(5,4)`, đường `(5,4)–(8,1)`.

**Ví dụ 2:** `[[3,4],[1,2],[7,8],[2,3]]` → `1` (sau khi sắp theo ngày, mọi điểm nằm trên `y = x + 1`).

**Ràng buộc:** `1 ≤ n ≤ 10⁵`, `1 ≤ day_i, price_i ≤ 10⁹`, các `day_i` phân biệt (input **chưa chắc** đã sắp xếp).

In [1]:
from typing import List
from fractions import Fraction
from math import gcd
import random, time

examples = [([[1, 7], [2, 6], [3, 5], [4, 4], [5, 4], [6, 3], [7, 2], [8, 1]], 3), ([[3, 4], [1, 2], [7, 8], [2, 3]], 1)]
def plot(points, H=8):
    pts = sorted(points); ys = {x: y for x, y in pts}
    xs = [x for x, _ in pts]; top = max(y for _, y in pts); bot = min(y for _, y in pts)
    for y in range(top, bot - 1, -1):
        print(f"{y:>3} | " + " ".join("*" if ys.get(x) == y else " " for x in range(xs[0], xs[-1] + 1)))
    print("      " + " ".join(str(x % 10) for x in range(xs[0], xs[-1] + 1)))
plot(examples[0][0])

  7 | *              
  6 |   *            
  5 |     *          
  4 |       * *      
  3 |           *    
  2 |             *  
  1 |               *
      1 2 3 4 5 6 7 8


## 3. Kiến thức nền

### 3.1 Sắp xếp điểm theo ngày

"Liền kề" nghĩa là liền kề **theo ngày** ⇒ phải `sort` trước (ví dụ 2 chưa sắp xếp).

In [2]:
print(sorted([[3, 4], [1, 2], [7, 8], [2, 3]]))

[[1, 2], [2, 3], [3, 4], [7, 8]]


### 3.2 So sánh hai hệ số góc bằng nhân chéo

Đoạn 1: `(dx₁, dy₁)`, đoạn 2: `(dx₂, dy₂)` (với `dx > 0` vì ngày tăng). Cùng hệ số góc ⇔ `dy₁/dx₁ = dy₂/dx₂` ⇔ **`dy₁ · dx₂ = dy₂ · dx₁`** — chỉ dùng số nguyên, chính xác tuyệt đối. Python có số nguyên tùy ý nên tích `10⁹ · 10⁹ = 10¹⁸` không tràn (C++ cần `long long`, vẫn vừa vì < 9,2·10¹⁸).

In [3]:
def same_slope(d1, d2):
    return d1[1] * d2[0] == d2[1] * d1[0]
print(same_slope((1, -1), (3, -3)), same_slope((1, -1), (1, 0)), same_slope((2, 1), (4, 2)))

True False True


### 3.3 Vì sao `float` không đủ chính xác

`float` có 53 bit mantissa (~15–16 chữ số thập phân). Hai hệ số góc **khác nhau** như `999999999/999999998` và `999999998/999999997` khác nhau cỡ `10⁻¹⁸` — nhỏ hơn độ phân giải của `float` quanh 1 (`~2,2·10⁻¹⁶`) ⇒ làm tròn thành **cùng** một số.

In [4]:
a = 999999999 / 999999998
b = 999999998 / 999999997
print(a, b, a == b, "| chính xác:", Fraction(999999999, 999999998) == Fraction(999999998, 999999997))
print("nhân chéo:", 999999999 * 999999997 == 999999998 * 999999998)

1.000000001 1.000000001 True | chính xác: False
nhân chéo: False


### 3.4 Phân số tối giản bằng `gcd`

`(dy, dx)` chia cho `g = gcd(|dy|, dx)` cho biểu diễn **duy nhất** của hệ số góc (vì `dx > 0`). So sánh tuple số nguyên là chính xác.

In [5]:
def norm(dx, dy):
    g = gcd(abs(dy), dx)
    return (dy // g, dx // g)
print(norm(3, -3), norm(1, -1), norm(4, 2), norm(2, 1), norm(5, 0))

(-1, 1) (-1, 1) (1, 2) (1, 2) (0, 1)


## 4. Ý tưởng nền tảng / trực giác

Sau khi sắp xếp theo ngày, biểu đồ là dãy `n − 1` đoạn nối liền nhau. Một "đường" (line) là một chuỗi đoạn **liên tiếp cùng hệ số góc** (các điểm thẳng hàng). Mỗi khi hệ số góc **thay đổi** giữa hai đoạn kề nhau, ta cần thêm một đường mới.

Đáp án = `1 + (số lần đổi hệ số góc)` (với `n ≥ 2`), hoặc 0 nếu `n = 1`.

Tính tay ví dụ 1 (đoạn và hệ số góc):

| đoạn | (dx, dy) | hệ số góc | đổi so với đoạn trước? |
|---|---|---|---|
| 1→2 | (1, −1) | −1 | — (đường 1) |
| 2→3 | (1, −1) | −1 | không |
| 3→4 | (1, −1) | −1 | không |
| 4→5 | (1, 0) | 0 | **có** (đường 2) |
| 5→6 | (1, −1) | −1 | **có** (đường 3) |
| 6→7, 7→8 | (1, −1) | −1 | không |

Đáp án 3. Chú ý: đường 3 có **cùng** hệ số góc với đường 1 nhưng **không** gộp được vì không liền kề (bị đoạn ngang chen giữa) — ta đếm **lần đổi**, không đếm **số hệ số góc khác nhau**.

In [6]:
pts = sorted(examples[0][0])
segs = [(x2 - x1, y2 - y1) for (x1, y1), (x2, y2) in zip(pts, pts[1:])]
print("các đoạn (dx, dy):", segs)
changes = sum(1 for d1, d2 in zip(segs, segs[1:]) if not same_slope(d1, d2))
print("số lần đổi hệ số góc:", changes, "-> đáp án", 1 + changes)

các đoạn (dx, dy): [(1, -1), (1, -1), (1, -1), (1, 0), (1, -1), (1, -1), (1, -1)]
số lần đổi hệ số góc: 2 -> đáp án 3


### 4.1 Hai hệ số góc khác nhau có thể gần nhau tới mức nào?

Với `a/b ≠ c/d` (các mẫu dương), `|a/b − c/d| = |ad − bc|/(bd) ≥ 1/(bd)`. Mẫu ≤ 10⁹ ⇒ hiệu ≥ 10⁻¹⁸. Trong khi đó khoảng cách giữa hai `float` liền kề quanh 1 là `2⁻⁵² ≈ 2,2·10⁻¹⁶` — **lớn hơn** 10⁻¹⁸ tới 200 lần. Vì vậy, về nguyên tắc, `float` **không thể** phân biệt mọi cặp hệ số góc trong ràng buộc này. Ngược lại, nhân chéo chỉ cần số nguyên ≤ `10⁹ · 10⁹ = 10¹⁸ < 2⁶³` — vừa với `long long`.

In [7]:
import math
print("khoảng cách float quanh 1:", math.ulp(1.0))
print("hiệu nhỏ nhất có thể giữa hai phân số khác nhau (mẫu <= 1e9):", 1 / (10**9 * 10**9))
print("tích lớn nhất khi nhân chéo:", 10**9 * 10**9, "< 2^63 - 1 =", 2**63 - 1)

khoảng cách float quanh 1: 2.220446049250313e-16
hiệu nhỏ nhất có thể giữa hai phân số khác nhau (mẫu <= 1e9): 1e-18
tích lớn nhất khi nhân chéo: 1000000000000000000 < 2^63 - 1 = 9223372036854775807


## 5. Cách 1 — Brute Force / baseline: gộp tham lam bằng kiểm tra thẳng hàng với mọi điểm của đường hiện tại

**Ý tưởng:** đi qua các điểm theo ngày, giữ danh sách điểm của "đường hiện tại"; điểm mới được thêm nếu nó **thẳng hàng với mọi điểm** của đường (kiểm tra với điểm đầu và điểm thứ hai là đủ, nhưng baseline cố tình kiểm tra hết). Nếu không, bắt đầu đường mới từ điểm cuối.

In [8]:
def collinear(a, b, c):
    return (b[0] - a[0]) * (c[1] - a[1]) == (c[0] - a[0]) * (b[1] - a[1])

def brute(stockPrices):
    pts = sorted(stockPrices)
    if len(pts) == 1: return 0
    lines = 1; cur = [pts[0], pts[1]]
    for p in pts[2:]:
        if all(collinear(cur[0], cur[1], q) for q in cur + [p]):
            cur.append(p)
        else:
            lines += 1; cur = [cur[-1], p]
    return lines

for sp, exp in examples:
    got = brute(sp)
    print(got, "OK" if got == exp else "SAI")

3 OK
1 OK


**Độ phức tạp:** worst O(n²) (đường dài: mỗi điểm mới kiểm tra lại toàn bộ đường). Đúng nhưng thừa.

**Nút thắt:** để biết điểm mới có nằm trên đường hiện tại, chỉ cần so **hệ số góc của đoạn mới** với **hệ số góc của đoạn trước** (thẳng hàng có tính bắc cầu theo đoạn liền kề). Một phép so sánh O(1).

## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** ba điểm liên tiếp `A, B, C` (theo ngày) cùng nằm trên một đường ⇔ hệ số góc `AB` = hệ số góc `BC` ⇔ `(B−A) × (C−B) = 0` (tích chéo). Vậy đáp án = `1 + #{i : đoạn i và đoạn i+1 khác hệ số góc}`.

**Quy tắc:**
- **Nếu** `n = 1` **thì** 0.
- Duyệt các bộ ba liên tiếp; **nếu** `dy₁·dx₂ ≠ dy₂·dx₁` **thì** `ans += 1`.
- Khởi tạo `ans = 1` cho đoạn đầu tiên.

**Chính xác số học:** **tuyệt đối không** dùng `dy/dx` kiểu float — mục 3.3 và Lỗi 1 cho thấy nó sai. Dùng nhân chéo (nhanh nhất), hoặc `Fraction`/tuple tối giản (chậm hơn nhưng cũng đúng).

**Bất biến:** sau khi xét đoạn `i`, `ans` = số đường tối thiểu cho các điểm `0..i+1`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Nhân chéo trên các đoạn liền kề (cách chuẩn)

In [9]:
def cross_mult(stockPrices):
    pts = sorted(stockPrices); n = len(pts)
    if n == 1: return 0
    ans = 1
    dx1, dy1 = pts[1][0] - pts[0][0], pts[1][1] - pts[0][1]
    for i in range(2, n):
        dx2, dy2 = pts[i][0] - pts[i - 1][0], pts[i][1] - pts[i - 1][1]
        if dy1 * dx2 != dy2 * dx1:
            ans += 1
        dx1, dy1 = dx2, dy2
    return ans

for sp, exp in examples:
    got = cross_mult(sp)
    print(got, "OK" if got == exp else "SAI")

3 OK
1 OK


**Độ phức tạp cách 2:** O(n log n) (sắp xếp) + O(n), O(n) bộ nhớ.

### 7.2 Cách 3 — Tích chéo của bộ ba điểm (dạng doocs)

In [10]:
def cross_triples(stockPrices):
    pts = sorted(stockPrices)
    if len(pts) < 2: return 0
    ans = 1
    for a, b, c in zip(pts, pts[1:], pts[2:]):
        if (b[1] - a[1]) * (c[0] - b[0]) != (c[1] - b[1]) * (b[0] - a[0]):
            ans += 1
    return ans

for sp, exp in examples:
    print(cross_triples(sp), "OK" if cross_triples(sp) == exp else "SAI")

3 OK
1 OK


### 7.3 Cách 4 — Hệ số góc dạng phân số tối giản (tuple)

In [11]:
def reduced_slopes(stockPrices):
    pts = sorted(stockPrices)
    if len(pts) < 2: return 0
    slopes = [norm(x2 - x1, y2 - y1) for (x1, y1), (x2, y2) in zip(pts, pts[1:])]
    return 1 + sum(1 for s1, s2 in zip(slopes, slopes[1:]) if s1 != s2)

for sp, exp in examples:
    print(reduced_slopes(sp), "OK" if reduced_slopes(sp) == exp else "SAI")

3 OK
1 OK


**Độ phức tạp cách 4:** O(n log n + n log V) (mỗi `gcd` O(log V)). Ưu điểm: hệ số góc thành **khóa băm được** — dùng được cho biến thể "đếm số hệ số góc khác nhau".

### 7.4 Cách 5 — `fractions.Fraction`

In [12]:
def with_fraction(stockPrices):
    pts = sorted(stockPrices)
    if len(pts) < 2: return 0
    slopes = [Fraction(y2 - y1, x2 - x1) for (x1, y1), (x2, y2) in zip(pts, pts[1:])]
    return 1 + sum(1 for s1, s2 in zip(slopes, slopes[1:]) if s1 != s2)

for sp, exp in examples:
    print(with_fraction(sp), "OK" if with_fraction(sp) == exp else "SAI")

3 OK
1 OK


`Fraction` chính xác nhưng chậm (mỗi phép tạo gọi `gcd` và tạo đối tượng). Dùng để **kiểm chứng**.

### 7.5 Cách 6 — `Decimal` độ chính xác cao (để thấy "tăng độ chính xác" chỉ là chữa cháy)

Với `getcontext().prec = 50`, phép chia `Decimal` đủ chính xác để phân biệt các hệ số góc trong ràng buộc này (hiệu tối thiểu giữa hai phân số khác nhau với mẫu ≤ 10⁹ là ≥ 10⁻¹⁸). Nhưng đây là cách vá, không phải cách nghĩ đúng.

In [13]:
from decimal import Decimal, getcontext
def with_decimal(stockPrices):
    getcontext().prec = 50
    pts = sorted(stockPrices)
    if len(pts) < 2: return 0
    slopes = [Decimal(y2 - y1) / Decimal(x2 - x1) for (x1, y1), (x2, y2) in zip(pts, pts[1:])]
    return 1 + sum(1 for s1, s2 in zip(slopes, slopes[1:]) if s1 != s2)

for sp, exp in examples:
    print(with_decimal(sp), "OK" if with_decimal(sp) == exp else "SAI")

3 OK
1 OK


## 8. TRACE từng bước

Trace cách 2 trên ví dụ 1.

In [14]:
def trace(stockPrices):
    pts = sorted(stockPrices); n = len(pts)
    print("điểm đã sắp:", pts)
    if n == 1: print("một điểm -> 0"); return 0
    ans = 1; dx1, dy1 = pts[1][0] - pts[0][0], pts[1][1] - pts[0][1]
    print(f"đoạn 0: (dx, dy) = ({dx1}, {dy1}) -> bắt đầu đường 1")
    for i in range(2, n):
        dx2, dy2 = pts[i][0] - pts[i - 1][0], pts[i][1] - pts[i - 1][1]
        lhs, rhs = dy1 * dx2, dy2 * dx1; diff = lhs != rhs
        if diff: ans += 1
        print(f"đoạn {i-1}: ({dx2}, {dy2}); so {dy1}*{dx2}={lhs} với {dy2}*{dx1}={rhs} -> {'ĐỔI, đường ' + str(ans) if diff else 'cùng đường'}")
        dx1, dy1 = dx2, dy2
    return ans

print("Kết quả:", trace(examples[0][0]))

điểm đã sắp: [[1, 7], [2, 6], [3, 5], [4, 4], [5, 4], [6, 3], [7, 2], [8, 1]]
đoạn 0: (dx, dy) = (1, -1) -> bắt đầu đường 1
đoạn 1: (1, -1); so -1*1=-1 với -1*1=-1 -> cùng đường
đoạn 2: (1, -1); so -1*1=-1 với -1*1=-1 -> cùng đường
đoạn 3: (1, 0); so -1*1=-1 với 0*1=0 -> ĐỔI, đường 2
đoạn 4: (1, -1); so 0*1=0 với -1*1=-1 -> ĐỔI, đường 3
đoạn 5: (1, -1); so -1*1=-1 với -1*1=-1 -> cùng đường
đoạn 6: (1, -1); so -1*1=-1 với -1*1=-1 -> cùng đường
Kết quả: 3


Trace ví dụ 2 (input chưa sắp xếp):

In [15]:
print("Kết quả:", trace(examples[1][0]))

điểm đã sắp: [[1, 2], [2, 3], [3, 4], [7, 8]]
đoạn 0: (dx, dy) = (1, 1) -> bắt đầu đường 1
đoạn 1: (1, 1); so 1*1=1 với 1*1=1 -> cùng đường
đoạn 2: (4, 4); so 1*4=4 với 4*1=4 -> cùng đường
Kết quả: 1


## 9. Code nộp LeetCode

In [16]:
from typing import List

class Solution:
    def minimumLines(self, stockPrices: List[List[int]]) -> int:
        pts = sorted(stockPrices)                    # nối theo thứ tự ngày
        n = len(pts)
        if n == 1:
            return 0
        ans = 1
        dx1 = pts[1][0] - pts[0][0]
        dy1 = pts[1][1] - pts[0][1]
        for i in range(2, n):
            dx2 = pts[i][0] - pts[i - 1][0]
            dy2 = pts[i][1] - pts[i - 1][1]
            if dy1 * dx2 != dy2 * dx1:               # nhân chéo: so hệ số góc chính xác
                ans += 1
            dx1, dy1 = dx2, dy2
        return ans

In [17]:
sol = Solution()
for sp, exp in examples:
    got = sol.minimumLines(sp)
    print(got, "| kỳ vọng", exp, "| OK" if got == exp else "| SAI")

3 | kỳ vọng 3 | OK
1 | kỳ vọng 1 | OK


## 10. Vì sao thuật toán đúng

Sau khi sắp xếp, biểu đồ là đường gấp khúc `P₀P₁…P_{n−1}` với `x` tăng ngặt.

**Phân hoạch thành đường thẳng.** Một tập các đoạn **liên tiếp** `P_aP_{a+1}, …, P_{b−1}P_b` nằm trên một đường thẳng ⇔ mọi cặp đoạn kề nhau trong đó cùng hệ số góc (vì hai đoạn kề nhau chung điểm `P_k`: cùng hệ số góc ⇔ ba điểm thẳng hàng; thẳng hàng lan truyền theo chuỗi). Đường biểu diễn phải chia dãy `n − 1` đoạn thành các **khối liên tiếp** như vậy; số khối ít nhất = số "ranh giới đổi hệ số góc" + 1 (mỗi ranh giới bắt buộc tách khối; ngoài ranh giới thì gộp được).

**Không gộp được hai khối không liền kề** dù cùng hệ số góc: đường biểu diễn phải đi qua mọi điểm theo thứ tự; hai đoạn song song tách biệt không thể là một đường (trừ khi trùng đường thẳng — nhưng khi đó các đoạn ở giữa cũng phải nằm trên đường đó, mâu thuẫn với việc có ranh giới).

**So sánh chính xác.** `dy₁/dx₁ = dy₂/dx₂ ⇔ dy₁dx₂ = dy₂dx₁` vì `dx₁, dx₂ > 0` (nhân hai vế với `dx₁dx₂ > 0` không đổi đẳng thức). ∎

**Hệ quả cho cách cài đặt:** thuật toán chỉ cần nhớ đoạn **ngay trước** — không cần nhớ "đường hiện tại" gồm những điểm nào, vì thẳng hàng giữa các đoạn liền kề có tính bắc cầu: nếu `P₀P₁ ∥ P₁P₂` và `P₁P₂ ∥ P₂P₃` thì cả bốn điểm thẳng hàng. Đây là lý do cách 1 (kiểm tra mọi điểm của đường) là thừa.

**Tại sao không cần quan tâm tới đoạn có `dx = 0`?** Ngày phân biệt và đã sắp tăng ⇒ `dx ≥ 1` luôn; đường thẳng đứng không thể xuất hiện. (Nếu ngày có thể trùng, cần thêm quy ước — biến thể ngoài phạm vi.)

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space | Chính xác? |
|---|---|---|---|
| 1. Gộp kiểm tra mọi điểm | O(n²) worst | O(n) | ✔ |
| 2. Nhân chéo đoạn kề | O(n log n) | O(n) | ✔ |
| 3. Tích chéo bộ ba | O(n log n) | O(n) | ✔ |
| 4. Phân số tối giản | O(n log n + n log V) | O(n) | ✔ |
| 5. Fraction | O(n log n), hằng số lớn | O(n) | ✔ |
| 6. Decimal prec 50 | O(n log n), hằng số lớn | O(n) | ✔ (vá) |
| Float `dy/dx` | O(n log n) | O(n) | ✘ (Lỗi 1) |

In [18]:
rng = random.Random(2280)
days = rng.sample(range(1, 10**9), 10**5)
big = [[d, rng.randint(1, 10**9)] for d in days]
line = [[d, 3 * d + 7] for d in range(1, 10**5 + 1)]           # một đường thẳng duy nhất
for data, nm in [(big, "ngẫu nhiên"), (line, "thẳng hàng")]:
    res = []
    for name, f in [("cross_mult", cross_mult), ("cross_triples", cross_triples), ("reduced_slopes", reduced_slopes),
                    ("with_fraction", with_fraction), ("Solution", Solution().minimumLines)]:
        t0 = time.perf_counter(); res.append(f(data)); dt = time.perf_counter() - t0
        print(f"{nm:<10} {name:<15} -> {res[-1]:<6} {dt*1000:6.0f} ms")
    assert len(set(res)) == 1

ngẫu nhiên cross_mult      -> 99999      88 ms


ngẫu nhiên cross_triples   -> 99999      93 ms
ngẫu nhiên reduced_slopes  -> 99999     126 ms


ngẫu nhiên with_fraction   -> 99999     149 ms
ngẫu nhiên Solution        -> 99999      80 ms
thẳng hàng cross_mult      -> 1          12 ms
thẳng hàng cross_triples   -> 1          11 ms
thẳng hàng reduced_slopes  -> 1          56 ms
thẳng hàng with_fraction   -> 1          59 ms
thẳng hàng Solution        -> 1          11 ms


## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [19]:
ALL = {"brute": brute, "cross_mult": cross_mult, "cross_triples": cross_triples, "reduced_slopes": reduced_slopes,
       "with_fraction": with_fraction, "with_decimal": with_decimal, "Solution": Solution().minimumLines}
edge = examples + [
    ([[1, 1]], 0),
    ([[1, 1], [2, 5]], 1),
    ([[1, 1], [2, 2], [3, 1]], 2),                       # lên rồi xuống
    ([[1, 1], [2, 2], [4, 4], [3, 3]], 1),               # chưa sắp xếp nhưng thẳng hàng
    ([[1, 1], [2, 2], [3, 3], [4, 2], [5, 1], [6, 2], [7, 3]], 3),   # đường 1 và 3 song song nhưng tách rời
    ([[1, 1], [500000000, 499999999], [1000000000, 999999999]], 2),   # gần thẳng hàng: float sẽ sai
    ([[1, 10**9], [10**9, 1]], 1),
]
for sp, exp in edge:
    for name, f in ALL.items():
        a = [list(p) for p in sp]
        assert f(a) == exp, (name, sp, f(sp), exp)
        assert a == sp, (name, "đã sửa input")
print("Edge case OK:", len(edge))

Edge case OK: 9


Case `[[1,1],[500000000,499999999],[1000000000,999999999]]`: hệ số góc đoạn 1 là `499999998/499999999`, đoạn 2 là `500000000/500000000 = 1` — khác nhau ⇒ 2 đường. Kiểm tra bằng nhân chéo: `499999998·500000000 ≠ 500000000·499999999`. Phép chia float có thể vẫn phân biệt được ở đây; Lỗi 1 cho một case float chắc chắn sai.

### 12.2 Đối chiếu ngẫu nhiên với brute force (seed cố định)

Sinh dữ liệu nhiều đoạn thẳng hàng (để có gộp) bằng cách đi theo vài hệ số góc ngẫu nhiên.

In [20]:
def gen_chart(rng, n):
    x, y = rng.randint(1, 5), rng.randint(1, 50); pts = [[x, y]]
    while len(pts) < n:
        dx, dy = rng.randint(1, 3), rng.randint(-3, 3)
        for _ in range(rng.randint(1, 4)):
            if len(pts) == n: break
            x += dx; y += dy; pts.append([x, y])
    rng.shuffle(pts); return pts

rng = random.Random(42)
for _ in range(1500):
    sp = gen_chart(rng, rng.randint(1, 25))
    ref = brute(sp)
    for name, f in ALL.items():
        assert f([list(p) for p in sp]) == ref, (name, sp)
print("1500 case ngẫu nhiên OK")

1500 case ngẫu nhiên OK


### 12.3 Dữ liệu lớn "gần thẳng hàng": đối chiếu nhân chéo với Fraction

In [21]:
rng = random.Random(7)
for _ in range(200):
    n = rng.randint(3, 40); xs = sorted(rng.sample(range(1, 10**9), n))
    a, b = rng.randint(1, 10**3), rng.randint(1, 10**3)
    sp = [[x, (a * x) // b + rng.choice([0, 0, 0, 1])] for x in xs]     # gần đường thẳng a/b
    assert Solution().minimumLines(sp) == with_fraction(sp) == reduced_slopes(sp)
print("gần thẳng hàng: nhân chéo khớp Fraction")

gần thẳng hàng: nhân chéo khớp Fraction


### 12.4 Cực trị theo ràng buộc

- `10⁵` điểm trên một đường thẳng với tọa độ lớn ⇒ 1.
- Răng cưa `(i, 1), (i+1, 2), (i+2, 1), …` ⇒ mỗi đoạn đổi hướng ⇒ `n − 1` đường (tối đa).

In [22]:
n = 10**5
straight = [[10**4 * i + 1, 7 * i + 3] for i in range(n)]
zigzag = [[i + 1, 1 + (i % 2)] for i in range(n)]
for name, f in ALL.items():
    if name in ("brute", "with_decimal", "with_fraction"): continue
    assert f(straight) == 1 and f(zigzag) == n - 1
print("cực trị OK: thẳng ->", Solution().minimumLines(straight), "| răng cưa ->", Solution().minimumLines(zigzag))

cực trị OK: thẳng -> 1 | răng cưa -> 99999


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — So sánh hệ số góc bằng `float` (bẫy nổi tiếng)

In [23]:
def wrong_float(stockPrices):
    pts = sorted(stockPrices)
    if len(pts) < 2: return 0
    slopes = [(y2 - y1) / (x2 - x1) for (x1, y1), (x2, y2) in zip(pts, pts[1:])]
    return 1 + sum(1 for s1, s2 in zip(slopes, slopes[1:]) if s1 != s2)

sp = [[1, 1], [500000000, 499999999], [1000000000, 999999998]]
(x1, y1), (x2, y2), (x3, y3) = sp
print("hệ số góc float  :", (y2 - y1) / (x2 - x1), (y3 - y2) / (x3 - x2), "-> bằng nhau?", (y2 - y1) / (x2 - x1) == (y3 - y2) / (x3 - x2))
print("hệ số góc chính xác:", Fraction(y2 - y1, x2 - x1), Fraction(y3 - y2, x3 - x2))
print("bản sai :", wrong_float(sp), "| bản đúng:", Solution().minimumLines(sp))

hệ số góc float  : 0.999999998 0.999999998 -> bằng nhau? True
hệ số góc chính xác: 499999998/499999999 499999999/500000000
bản sai : 1 | bản đúng: 2


**Vì sao sai:** hai hệ số góc `499999998/499999999` và `499999999/500000000` gần bằng 1 và khác nhau cỡ 4·10⁻¹⁸ được làm tròn thành cùng một `float` ⇒ bản sai gộp hai đoạn **không** thẳng hàng. **Sửa:** nhân chéo số nguyên.

### Lỗi 2 — Không sắp xếp theo ngày

In [24]:
def wrong_no_sort(stockPrices):
    pts = stockPrices                                   # SAI
    if len(pts) < 2: return 0
    ans = 1
    for a, b, c in zip(pts, pts[1:], pts[2:]):
        if (b[1] - a[1]) * (c[0] - b[0]) != (c[1] - b[1]) * (b[0] - a[0]): ans += 1
    return ans

print("ví dụ 2 - bản sai :", wrong_no_sort([[3, 4], [1, 2], [7, 8], [2, 3]]), "| bản đúng:", Solution().minimumLines([[3, 4], [1, 2], [7, 8], [2, 3]]), "(trùng hợp: mọi điểm thẳng hàng)")
shuffled = [[4, 4], [1, 7], [8, 1], [2, 6], [5, 4], [3, 5], [7, 2], [6, 3]]
print("ví dụ 1 xáo trộn - bản sai :", wrong_no_sort(shuffled), "| bản đúng:", Solution().minimumLines(shuffled))

ví dụ 2 - bản sai : 1 | bản đúng: 1 (trùng hợp: mọi điểm thẳng hàng)
ví dụ 1 xáo trộn - bản sai : 7 | bản đúng: 3


Với ví dụ 2, mọi điểm nằm trên một đường thẳng nên thứ tự nào cũng "thẳng hàng" — bản sai tình cờ đúng. Xáo trộn ví dụ 1 thì lộ lỗi: các "đoạn" nối hai điểm không liền kề theo ngày.

**Sửa:** `sorted(stockPrices)` trước khi nối.

### Lỗi 3 — Đếm **số hệ số góc khác nhau** thay vì số lần đổi

In [25]:
def wrong_distinct_slopes(stockPrices):
    pts = sorted(stockPrices)
    if len(pts) < 2: return 0
    return len({norm(x2 - x1, y2 - y1) for (x1, y1), (x2, y2) in zip(pts, pts[1:])})    # SAI

print("bản sai :", wrong_distinct_slopes(examples[0][0]), "| bản đúng:", Solution().minimumLines(examples[0][0]))

bản sai : 2 | bản đúng: 3


**Vì sao sai:** ví dụ 1 có hai hệ số góc khác nhau (−1 và 0) nhưng cần 3 đường vì đoạn hệ số góc −1 bị tách làm hai bởi đoạn ngang. **Sửa:** đếm **ranh giới** giữa các đoạn liền kề.

### Lỗi 4 — Quên trường hợp một điểm (trả 1 thay vì 0)

In [26]:
def wrong_single(stockPrices):
    pts = sorted(stockPrices); ans = 1                 # SAI khi n = 1
    for a, b, c in zip(pts, pts[1:], pts[2:]):
        if (b[1] - a[1]) * (c[0] - b[0]) != (c[1] - b[1]) * (b[0] - a[0]): ans += 1
    return ans

print("bản sai :", wrong_single([[5, 5]]), "| bản đúng:", Solution().minimumLines([[5, 5]]))

bản sai : 1 | bản đúng: 0


**Sửa:** `n == 1` ⇒ 0 (không có đoạn nào để vẽ).

### Lỗi 5 — "Sửa" float bằng epsilon

In [27]:
def wrong_epsilon(stockPrices, eps=1e-9):
    pts = sorted(stockPrices)
    if len(pts) < 2: return 0
    slopes = [(y2 - y1) / (x2 - x1) for (x1, y1), (x2, y2) in zip(pts, pts[1:])]
    return 1 + sum(1 for s1, s2 in zip(slopes, slopes[1:]) if abs(s1 - s2) > eps)

sp = [[1, 1], [500000000, 499999999], [1000000000, 999999998]]   # hai hệ số góc khác nhau ~4e-18
print("eps=1e-9  bản sai :", wrong_epsilon(sp), "| bản đúng:", Solution().minimumLines(sp))
print("eps=1e-20 bản sai :", wrong_epsilon(sp, 1e-20), "| bản đúng:", Solution().minimumLines(sp), "(float đã làm tròn hai số thành một)")

eps=1e-9  bản sai : 1 | bản đúng: 2
eps=1e-20 bản sai : 1 | bản đúng: 2 (float đã làm tròn hai số thành một)


**Vì sao sai:** khoảng cách nhỏ nhất giữa hai hệ số góc **khác nhau** có mẫu ≤ 10⁹ có thể nhỏ cỡ 10⁻¹⁸ — nhỏ hơn mọi epsilon hợp lý và nhỏ hơn cả độ phân giải `float`. Không có epsilon nào vừa phân biệt được mọi cặp khác nhau vừa chịu được sai số làm tròn. **Sửa:** so sánh chính xác bằng số nguyên.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- So sánh hệ số góc / thẳng hàng ⇒ **nhân chéo số nguyên**; **không bao giờ** so sánh `float` khi tọa độ lớn.
- "Số đoạn ít nhất" của đường gấp khúc ⇒ `1 + số điểm gãy` (sau khi sắp xếp).
- Phân biệt "số lần đổi" (ranh giới) với "số giá trị khác nhau".
- Biên: `n = 1` ⇒ 0.
- Khi thấy tọa độ tới 10⁹ trong bài hình học, mặc định nghĩ ngay: "**không float**, dùng nhân chéo hoặc phân số tối giản".
- Bài cùng họ: 1232 Check If It Is a Straight Line, 149 Max Points on a Line (khóa phân số tối giản), 2152 Minimum Number of Lines to Cover Points, 1037 Valid Boomerang.

## 15. Biến thể của bài

### 15.1 Biến thể A — Trả về **các đường** (cặp điểm đầu–cuối của mỗi đường)

In [28]:
def lines_endpoints(stockPrices):
    pts = sorted(stockPrices)
    if len(pts) < 2: return []
    out = []; start = pts[0]
    for a, b, c in zip(pts, pts[1:], pts[2:]):
        if (b[1] - a[1]) * (c[0] - b[0]) != (c[1] - b[1]) * (b[0] - a[0]):
            out.append((tuple(start), tuple(b))); start = b
    out.append((tuple(start), tuple(pts[-1])))
    return out

print(lines_endpoints(examples[0][0]))

[((1, 7), (4, 4)), ((4, 4), (5, 4)), ((5, 4), (8, 1))]


### 15.2 Biến thể B — Đếm số **đỉnh cục bộ** (điểm đổi từ tăng sang giảm)

Chỉ cần dấu của `dy` (vì `dx > 0`): đỉnh khi `dy₁ > 0` và `dy₂ < 0`.

In [29]:
def local_peaks(stockPrices):
    pts = sorted(stockPrices)
    return sum(1 for a, b, c in zip(pts, pts[1:], pts[2:]) if b[1] > a[1] and c[1] < b[1])

print(local_peaks([[1, 1], [2, 3], [3, 2], [4, 5], [5, 1]]))

2


### 15.3 Biến thể C — Cho phép **bỏ** tối đa một điểm để giảm số đường

Thử bỏ từng điểm (O(n²)) cho dữ liệu nhỏ; minh họa rằng một điểm "nhiễu" có thể làm tăng 2 đường.

In [30]:
def min_lines_remove_one(stockPrices):
    pts = sorted(stockPrices)
    best = Solution().minimumLines(pts)
    for i in range(len(pts)):
        rest = pts[:i] + pts[i + 1:]
        if rest: best = min(best, Solution().minimumLines(rest))
    return best

noisy = [[1, 1], [2, 2], [3, 9], [4, 4], [5, 5]]
print(Solution().minimumLines(noisy), "->", min_lines_remove_one(noisy))

4 -> 1


### 15.4 Biến thể D — Đếm số **hệ số góc khác nhau** (đúng là "Lỗi 3" — nhưng cho câu hỏi khác)

Dùng khóa phân số tối giản (cách 4) cho vào `set`. Đây là ví dụ cùng một đoạn code đúng cho bài này, sai cho bài kia.

In [31]:
def distinct_slopes(stockPrices):
    pts = sorted(stockPrices)
    return len({norm(x2 - x1, y2 - y1) for (x1, y1), (x2, y2) in zip(pts, pts[1:])})
print(distinct_slopes(examples[0][0]), "(số hệ số góc khác nhau) vs", Solution().minimumLines(examples[0][0]), "(số đường)")

2 (số hệ số góc khác nhau) vs 3 (số đường)


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Gộp kiểm tra mọi điểm | thẳng hàng với cả đường | O(n²) | O(n) | baseline |
| 2. Nhân chéo đoạn kề | `dy₁dx₂ ≠ dy₂dx₁` | O(n log n) | O(n) | **mặc định** |
| 3. Tích chéo bộ ba | `zip` ba dãy | O(n log n) | O(n) | gọn |
| 4. Phân số tối giản | tuple `(dy/g, dx/g)` | O(n log n) | O(n) | cần khóa băm |
| 5. Fraction | chính xác, chậm | O(n log n) | O(n) | kiểm chứng |
| 6. Decimal | tăng độ chính xác | O(n log n) | O(n) | không khuyến khích |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `minimumLines([[1,1],[2,2],[3,3],[4,5]])` và `minimumLines([[1,5],[2,5],[3,5]])`?

**Bài 2 (tính tay):** Với `[[1,2],[3,6],[4,7],[6,9],[7,8]]`, liệt kê `(dx, dy)` của các đoạn và các điểm gãy.

**Bài 3 (điền TODO):** Viết `bt3_todo(stockPrices)` trả về **danh sách ngày** của các điểm gãy (điểm mà hai đoạn kề khác hệ số góc).

**Bài 4 (tự viết lại):** Viết `bt4_todo(stockPrices)` cài cách 4 (phân số tối giản).

**Bài 5 (mini challenge):** Viết `bt5_todo(stockPrices)` trả về **độ dài lớn nhất** (số điểm) của một đường trong biểu đồ (một khối đoạn liên tiếp cùng hệ số góc).

In [32]:
def bt3_todo(stockPrices):
    # TODO: ngày của các điểm gãy
    pass

def bt4_todo(stockPrices):
    # TODO: phân số tối giản
    pass

def bt5_todo(stockPrices):
    # TODO: số điểm lớn nhất trên một đường của biểu đồ
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: đoạn `(1,1),(1,1),(1,2)`. Mức 2: đổi ở điểm ngày 3. Mức 3: `2`; ví dụ hai: mọi đoạn ngang ⇒ `1`.

**Bài 2.** Mức 1: đoạn `(2,4), (1,1), (2,2), (1,−1)`. Mức 2: hệ số góc `2, 1, 1, −1`. Mức 3: điểm gãy tại ngày 3 và ngày 6 ⇒ 3 đường.

**Bài 3.** Mức 1: duyệt bộ ba `(a, b, c)`. Mức 2: nếu tích chéo ≠ 0 thì `b` là điểm gãy. Mức 3: thu `b[0]`.

**Bài 4.** Mức 1: `norm(dx, dy)` với `gcd`. Mức 2: danh sách hệ số góc. Mức 3: đếm vị trí đổi + 1.

**Bài 5.** Mức 1: chia dãy đoạn thành khối cùng hệ số góc. Mức 2: khối có `k` đoạn chứa `k + 1` điểm. Mức 3: max (k + 1); `n = 1` ⇒ 1.

## 19. Đáp án tham khảo cho bài tập

**Bài 1:** `2`, `1`. **Bài 2:** đoạn `(2,4),(1,1),(2,2),(1,−1)`; điểm gãy ngày 3 và 6 ⇒ 3 đường.

In [33]:
assert Solution().minimumLines([[1, 1], [2, 2], [3, 3], [4, 5]]) == 2
assert Solution().minimumLines([[1, 5], [2, 5], [3, 5]]) == 1
assert Solution().minimumLines([[1, 2], [3, 6], [4, 7], [6, 9], [7, 8]]) == 3

def bt3(stockPrices):
    pts = sorted(stockPrices)
    return [b[0] for a, b, c in zip(pts, pts[1:], pts[2:])
            if (b[1] - a[1]) * (c[0] - b[0]) != (c[1] - b[1]) * (b[0] - a[0])]

def bt4(stockPrices):
    return reduced_slopes(stockPrices)

def bt5(stockPrices):
    pts = sorted(stockPrices)
    if len(pts) < 2: return len(pts)
    best = cur = 2
    for a, b, c in zip(pts, pts[1:], pts[2:]):
        if (b[1] - a[1]) * (c[0] - b[0]) == (c[1] - b[1]) * (b[0] - a[0]): cur += 1
        else: cur = 2
        best = max(best, cur)
    return best

def bt5_brute(stockPrices):
    pts = sorted(stockPrices); n = len(pts)
    if n < 2: return n
    best = 2
    for i in range(n):
        for j in range(i + 2, n + 1):
            seg = pts[i:j]
            if all(collinear(seg[0], seg[1], q) for q in seg): best = max(best, j - i)
    return best

assert bt3([[1, 2], [3, 6], [4, 7], [6, 9], [7, 8]]) == [3, 6]
rng = random.Random(1980)
for _ in range(800):
    sp = gen_chart(rng, rng.randint(1, 20))
    assert len(bt3(sp)) == max(0, Solution().minimumLines(sp) - 1)
    assert bt4(sp) == brute(sp)
    assert bt5(sp) == bt5_brute(sp)
print("Đáp án bài tập: tất cả assert OK")

Đáp án bài tập: tất cả assert OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
pts = sorted(stockPrices)
if len(pts) == 1: return 0
ans = 1
for a, b, c in zip(pts, pts[1:], pts[2:]):
    if (b[1]-a[1]) * (c[0]-b[0]) != (c[1]-b[1]) * (b[0]-a[0]):   # nhân chéo, KHÔNG float
        ans += 1
return ans
```

**Checklist:**
- [ ] Sắp xếp theo ngày.
- [ ] So hệ số góc bằng **nhân chéo** (tọa độ 10⁹ ⇒ float sai).
- [ ] Đếm **lần đổi** giữa đoạn liền kề, không đếm số hệ số góc khác nhau.
- [ ] `n = 1` ⇒ 0; `n = 2` ⇒ 1.
- [ ] (C++/Java) tích tới 10¹⁸ ⇒ `long long`/`long`.